In [ ]:
%pip install pandas astroquery pyvo

In [2]:
from astroquery.jplhorizons import Horizons
import pandas as pd
import numpy as np
import pyvo as vo

# Test generate coords for Eureka

In [3]:

start = '2018-01-01' # yyyy-mm-dd format 
stop = '2023-01-01'
step = '1d' # Step size

obj = Horizons(id='Eureka', location='Q55',
               epochs={'start':start, 'stop':stop,
                       'step':step})

In [4]:
eph = obj.ephemerides()

In [ ]:
# See type returned
print(eph.dtype)

[('targetname', '<U21'), ('datetime_str', '<U17'), ('datetime_jd', '<f8'), ('H', '<f8'), ('G', '<f8'), ('solar_presence', '<U1'), ('lunar_presence', '<U1'), ('RA', '<f8'), ('DEC', '<f8'), ('RA_app', '<f8'), ('DEC_app', '<f8'), ('RA_rate', '<f8'), ('DEC_rate', '<f8'), ('AZ', '<f8'), ('EL', '<f8'), ('AZ_rate', '<f8'), ('EL_rate', '<f8'), ('sat_X', '<f8'), ('sat_Y', '<f8'), ('sat_PANG', '<f8'), ('siderealtime', '<f8'), ('airmass', '<f8'), ('magextinct', '<f8'), ('V', '<f8'), ('surfbright', '<f8'), ('illumination', '<f8'), ('illum_defect', '<f8'), ('sat_sep', '<f8'), ('sat_vis', '<U1'), ('ang_width', '<f8'), ('PDObsLon', '<i8'), ('PDObsLat', '<i8'), ('PDSunLon', '<i8'), ('PDSunLat', '<i8'), ('SubSol_ang', '<f8'), ('SubSol_dist', '<f8'), ('NPole_ang', '<i8'), ('NPole_dist', '<i8'), ('EclLon', '<f8'), ('EclLat', '<f8'), ('r', '<f8'), ('r_rate', '<f8'), ('delta', '<f8'), ('delta_rate', '<f8'), ('lighttime', '<f8'), ('vel_sun', '<f8'), ('vel_obs', '<f8'), ('elong', '<f8'), ('elongFlag', '<U2')

# Connect to Skymapper via TAP

In [5]:
service = vo.dal.TAPService('https://api.skymapper.nci.org.au/public/tap')

In [6]:
# Test print table names

query = """
SELECT table_name
FROM TAP_SCHEMA.tables
"""

tables = service.search(query)

In [7]:
tablesdf = tables.to_table().to_pandas()
# List of tables
tablesdf.to_csv("tables.txt", index=False, sep="\t", header=0)

# Find start dates for image

In [8]:
from astropy.time import Time

In [9]:
query = """
SELECT DISTINCT night_mjd
FROM dr4.images
ORDER BY night_mjd ASC
"""

results = service.search(query)

In [10]:
datesdf = results.to_table().to_pandas()
datesdf.columns = ["time"]

In [11]:
datesdf["time"] = Time(datesdf["time"].values, format="mjd") # Turn into astropy time object (modified julian date)
datesdf["date"]  = datesdf["time"].apply(lambda t: t.to_value("iso", subfmt="date"))       # '2023-03-01 00:00:00.000'

In [12]:
print(datesdf)

         time        date
0     56730.0  2014-03-14
1     56731.0  2014-03-15
2     56732.0  2014-03-16
3     56733.0  2014-03-17
4     56734.0  2014-03-18
...       ...         ...
1752  59469.0  2021-09-12
1753  59470.0  2021-09-13
1754  59471.0  2021-09-14
1755  59472.0  2021-09-15
1756  59473.0  2021-09-16

[1757 rows x 2 columns]


## Test: Find the last image, and print out all ccds and their columns to this image

In [ ]:
# Select last image
query = """
SELECT TOP 1 image_id
FROM dr4.images
ORDER BY night_mjd DESC
"""
results = service.search(query)
results = results.to_table()

In [34]:
print(results)

   image_id   
--------------
20210916085152


In [38]:
img_id = results[0]["image_id"]
print(img_id) # Find image id of last image

20210916085152


In [ ]:
# Select ccds for last image
query = f"""
SELECT *
FROM dr4.ccds
WHERE image_id = {img_id}
"""
results = service.search(query)
results = results.to_table()

In [45]:
print(results.columns)
print(results.dtype)

for i in results[0]:
    print(type(i))

<TableColumns names=('image_id','ccd','image_type','filename','maskname','image','filter','mjd_obs','fwhm_ccd','elong_ccd','nsatpix','sb_mag','phot_nstar','header','coverage')>
[('image_id', '<i8'), ('ccd', '<i4'), ('image_type', '<U256'), ('filename', '<U256'), ('maskname', '<U256'), ('image', '<U256'), ('filter', '<U256'), ('mjd_obs', '<f8'), ('fwhm_ccd', '<f4'), ('elong_ccd', '<f4'), ('nsatpix', '<i4'), ('sb_mag', '<f4'), ('phot_nstar', '<i4'), ('header', '<U256'), ('coverage', '<U69')]
<class 'numpy.int64'>
<class 'numpy.int32'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.float64'>
<class 'numpy.float32'>
<class 'numpy.float32'>
<class 'numpy.int32'>
<class 'numpy.float32'>
<class 'numpy.int32'>
<class 'numpy.str_'>
<class 'numpy.str_'>


# Tests - explore photometry

In [ ]:
# Select all photometry for last image
query = f"""
SELECT TOP 1 *
FROM dr4.photometry
WHERE image_id = 20210916085152
"""
results = service.search(query)
results = results.to_table()


In [50]:
print(results.columns)
print(results.dtype)
print(results)

<TableColumns names=('object_id','object_id_local','image_id','ccd','filter','ra_img','decl_img','x_img','y_img','flags','nimaflags','imaflags','background','flux_max','mu_max','class_star','a','e_a','b','e_b','pa','e_pa','elong','fwhm','radius_petro','radius_kron','radius_frac20','radius_frac50','radius_frac90','chi2_psf','flux_psf','e_flux_psf','mag_psf','e_mag_psf','flux_petro','e_flux_petro','mag_petro','e_mag_petro','flux_kron','e_flux_kron','mag_kron','e_mag_kron','flux_ap02','e_flux_ap02','mag_apc02','e_mag_apc02','flux_ap03','e_flux_ap03','mag_apc03','e_mag_apc03','flux_ap04','e_flux_ap04','mag_apc04','e_mag_apc04','flux_ap05','e_flux_ap05','mag_apc05','e_mag_apc05','flux_ap06','e_flux_ap06','mag_apc06','e_mag_apc06','flux_ap08','e_flux_ap08','mag_apc08','e_mag_apc08','flux_ap10','e_flux_ap10','mag_apc10','e_mag_apc10','flux_ap15','e_flux_ap15','mag_apr15','e_mag_apr15','flux_ap20','e_flux_ap20','mag_apr20','e_mag_apr20','flux_ap30','e_flux_ap30','mag_apr30','e_mag_apr30','use_

In [13]:
# - Coordinate Generation - 
from astroquery.jplhorizons import Horizons
import pandas as pd 

# - Set dates based on Bradley's SkyMapper DR4 information -
start_date = '2014-03-14 12:00'
stop_date = '2021-09-17 12:00'
step = '1d'

print("Querying NASA Horizons... please wait.")

# - Setup the query for Eureka at Siding Spring (Q55) - 
obj = Horizons(id='5261', location='Q55',
               epochs={'start': start_date, 'stop': stop_date, 'step': step})

eph = obj.ephemerides()
df_raw = eph.to_pandas()

# - Apply Filters: -
df_clean = df_raw[df_raw['V'] < 24].copy()

# - Add MJD column for Bradley - 
# - Formula: MJD = Julian Date - 2400000.5
df_clean['mjd']=(df_clean['datetime_jd'] - 2400000.5).astype(int)

# - Save as CSV - 
df_clean.to_csv('eureka_roadmap.csv', index=False)

print(f"DONE! Created 'eureka_roadmap.csv' with {len(df_clean)} rows.")
print(f"Total columns: {len(df_clean.columns)} 
df_clean[['mjd', 'datetime_str', 'RA', 'DEC', 'V']].head()

SyntaxError: unterminated string literal (detected at line 27) (19023345.py, line 27)

In [11]:
# - Test point from data (row 0) - 
test_ra = 96.543
test_dec = -6.364
